In [1]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
datapath = '../data/2010-2011 Solar home electricity data.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_01"

In [2]:
datapath = '../data/2011-2012 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_02"

shape: (270_304, 54)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬─────────────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 23:00 ┆ 23:30 ┆ 0:00  ┆ Row Quality │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---         │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ str         │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆             │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═════════════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 1.063 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.118 ┆ 0.219 ┆ 0.162 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆

In [ ]:
datapath = '../data/2012-2013 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_03"

In [3]:
# we can get the training and testing customers from the csv file
training_customers = np.loadtxt('../data/training_customers.csv', dtype=int)
testing_customers = np.loadtxt('../data/testing_customers.csv', dtype=int)

In [ ]:
# alternatively, get all the unique customers as their own dataframes
customers = df['Customer'].unique()
# pick 80% of the random customers as training data
training_customers = np.random.choice(customers, int(0.8*len(customers)), replace=False)
# the rest of the customers are testing data
testing_customers = np.setdiff1d(customers, training_customers)

In [ ]:
# save the customers number to a csv file
np.savetxt('../data/training_customers.csv', training_customers, fmt='%s')
np.savetxt('../data/testing_customers.csv', testing_customers, fmt='%s')

In [4]:
from helper import transform_polars_df
# loop through each customer and use transform_polars_df to get the dataframe and store it in a list call dataset
training_dataset = []
for customer in training_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as training dataset: {customer}")
        print(e)
        break
    training_dataset.append(newcustomerdf)

testing_dataset = []
for customer in testing_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as testing dataset: {customer}")
        print(e)
        break
    testing_dataset.append(newcustomerdf)

In [ ]:
# provide std and mean on the different columns of the training dataset
testdf = testing_dataset[25]
# drop timestamp and time columns
testdf = testdf.drop(['Timestamp', 'Time'])
print(testdf.describe())

In [5]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env

testing_env_fns = [make_env(ds) for ds in testing_dataset]

num_step = None # pick the number of step for the simulation/none for full length
test_envs = [env_fn(num_step) for env_fn in testing_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [15]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env
# Create a list of environment creation functions to build a vectorized environment.
training_env_fns = [make_env(ds) for ds in training_dataset]
#training_vec_env = DummyVecEnv(training_env_fns)
num_step = None # pick the number of step for the simulation/none for full length
train_envs = [env_fn(num_step) for env_fn in training_env_fns]

In [ ]:
# combine the test_envs and train_envs into a single list
combined_envs = test_envs + train_envs

In [16]:
selected_list = train_envs
if selected_list is test_envs:
    env_type = "test"
    env_fns = testing_env_fns
elif selected_list is train_envs:
    env_type = "train"
    env_fns = training_env_fns
else:
    env_type = "combined"
    env_fns = testing_env_fns + training_env_fns

In [17]:
from decision import Agent, run_episodes_parallel
rule_agent_kwargs = {
    'algorithm': 'rule'
}

# run episodes in the list in parallel using the rule-based agent on the training environments
episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=rule_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 240 episodes with max_workers=12


Episodes: 100%|██████████| 240/240 [05:11<00:00,  1.30s/it]


[START] Episode 5
Sim Complete
[DONE]  Episode 5 (Elapsed: 16.71 sec)
[START] Episode 17
Sim Complete
[DONE]  Episode 17 (Elapsed: 16.84 sec)
[START] Episode 33
Sim Complete
[DONE]  Episode 33 (Elapsed: 16.75 sec)
[START] Episode 45
Sim Complete
[DONE]  Episode 45 (Elapsed: 17.74 sec)
[START] Episode 59
Sim Complete
[DONE]  Episode 59 (Elapsed: 18.36 sec)
[START] Episode 72
Sim Complete
[DONE]  Episode 72 (Elapsed: 15.98 sec)
[START] Episode 84
Sim Complete
[DONE]  Episode 84 (Elapsed: 15.88 sec)
[START] Episode 97
Sim Complete
[DONE]  Episode 97 (Elapsed: 16.31 sec)
[START] Episode 110
Sim Complete
[DONE]  Episode 110 (Elapsed: 16.23 sec)
[START] Episode 124
Sim Complete
[DONE]  Episode 124 (Elapsed: 15.87 sec)
[START] Episode 138
Sim Complete
[DONE]  Episode 138 (Elapsed: 16.16 sec)
[START] Episode 151
Sim Complete
[DONE]  Episode 151 (Elapsed: 16.27 sec)
[START] Episode 163
Sim Complete
[DONE]  Episode 163 (Elapsed: 15.93 sec)
[START] Episode 175
Sim Complete
[DONE]  Episode 175 (El

In [ ]:
# Collect all schemas
schemas = [df.schema for df in dfs_with_id]

# Find the most common schema (assume it's the correct one)
from collections import Counter
schema_counts = Counter([tuple(sorted(s.items())) for s in schemas])
most_common_schema = dict(schema_counts.most_common(1)[0][0])

# Print out indices and details of DataFrames with mismatched schemas
for i, schema in enumerate(schemas):
    if dict(sorted(schema.items())) != most_common_schema:
        print(f"DF {i} schema mismatch:")
        print("Schema:", schema)
        print("Difference:", set(schema.items()) ^ set(most_common_schema.items()))

In [18]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_logs)]
rule_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/rule_{env_type}_{episode_num}_logs.parquet"
rule_all_logs.write_parquet(file_name)

In [19]:
from decision import Agent, run_episodes_parallel, run_single
# Initialize environments and SDP agent parameters
sdp_agent_kwargs = {
    'algorithm': 'sdp',
    'soc_resolution': 20,
    'action_resolution': 41,  # best to be 2*soc_resolution + 1
    'degradation_model': 'static' # the other option being static degradation 'static'
    #'linear_deg_cost_p_kwh': 0.2
}

# Run a single episode for timing test
#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=sdp_agent_kwargs, render=False, display_progress=True)


# Run all episodes in parallel
sdp_episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=sdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 240 episodes with max_workers=12


Episodes:  11%|█▏        | 27/240 [1:32:03<29:13:16, 493.88s/it]

[START] Episode 1
Sim Complete
[DONE]  Episode 1 (Elapsed: 1880.01 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 1882.66 sec)
[START] Episode 28


Episodes:  12%|█▏        | 29/240 [1:34:49<16:23:58, 279.80s/it]

[START] Episode 5
Sim Complete
[DONE]  Episode 5 (Elapsed: 1911.56 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 1885.86 sec)
[START] Episode 29


...

Sim Complete
[DONE]  Episode 199 (Elapsed: 1938.83 sec)
[START] Episode 212
Sim Complete
[DONE]  Episode 212 (Elapsed: 1948.80 sec)
[START] Episode 225
Sim Complete
[DONE]  Episode 225 (Elapsed: 1959.94 sec)
[START] Episode 238


Episodes: 100%|█████████▉| 239/240 [10:15:28<02:26, 146.21s/it]

Sim Complete
[DONE]  Episode 201 (Elapsed: 1916.48 sec)
[START] Episode 213
Sim Complete
[DONE]  Episode 213 (Elapsed: 1942.88 sec)
[START] Episode 226
Sim Complete
[DONE]  Episode 226 (Elapsed: 1921.25 sec)
[START] Episode 239


Episodes: 100%|██████████| 240/240 [10:16:00<00:00, 154.00s/it]


Sim Complete
[DONE]  Episode 228 (Elapsed: 1890.79 sec)
Sim Complete
[DONE]  Episode 211 (Elapsed: 1919.45 sec)
[START] Episode 224
Sim Complete
[DONE]  Episode 224 (Elapsed: 986.49 sec)
[START] Episode 227
Sim Complete
[DONE]  Episode 227 (Elapsed: 1805.75 sec)
Sim Complete
[DONE]  Episode 229 (Elapsed: 1868.88 sec)
Sim Complete
[DONE]  Episode 205 (Elapsed: 1933.60 sec)
[START] Episode 217
Sim Complete
[DONE]  Episode 217 (Elapsed: 1916.55 sec)
[START] Episode 230
Sim Complete
[DONE]  Episode 230 (Elapsed: 1888.63 sec)
Sim Complete
[DONE]  Episode 216 (Elapsed: 1944.98 sec)
[START] Episode 231
Sim Complete
[DONE]  Episode 231 (Elapsed: 1895.69 sec)
[DONE]  Episode 218 (Elapsed: 1903.11 sec)
[START] Episode 232
Sim Complete
[DONE]  Episode 232 (Elapsed: 1860.12 sec)
Sim Complete
[DONE]  Episode 233 (Elapsed: 1853.34 sec)
Sim Complete
[DONE]  Episode 234 (Elapsed: 1869.58 sec)
Sim Complete
[DONE]  Episode 236 (Elapsed: 1853.25 sec)
Sim Complete
[DONE]  Episode 207 (Elapsed: 1930.41 sec

In [20]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(sdp_episode_logs)]
sdp_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/sdp_{env_type}_{episode_num}_logs.parquet"
sdp_all_logs.write_parquet(file_name)

In [21]:
from decision import Agent, run_episodes_parallel, run_single
mrdp_agent_kwargs = {
    'algorithm': 'mrdp',
    'soc_resolution': 20,           # fallback/default for single-horizon
    'action_resolution': 41,        # fallback/default for single-horizon
    'degradation_model': 'static',  # or 'static'
    #'linear_deg_cost_p_kwh': 0.2,   # only needed if using linear
    'subhorizon_specs': [
        {
            'start': 0,
            'length': 12,           # e.g. 6 hours at 30-min steps
            'soc_res': 20,          # fine SoC discretization
            'action_res': 11,       # fine action discretization
            'step_duration': 0.5    # hours per step (30 min)
        },
        {
            'start': 12,
            'length': 36,           # e.g. 18 hours at 30-min steps
            'soc_res': 8,           # coarse SoC discretization
            'action_res': 5,        # coarse action discretization
            'step_duration': 1.0    # hours per step (1 hour)
        }
    ]
}

#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=mrdp_agent_kwargs, render=False, display_progress=True)

# Run all episodes in parallel using MRDP

mrdp_episode_logs = run_episodes_parallel(
    Agent, selected_list, agent_kwargs=mrdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False
)

[INFO] Starting 240 episodes with max_workers=12


Episodes:  12%|█▏        | 28/240 [05:39<1:18:10, 22.12s/it]

[START] Episode 8
Sim Complete
[DONE]  Episode 8 (Elapsed: 114.78 sec)
[START] Episode 17
Sim Complete
[DONE]  Episode 17 (Elapsed: 111.54 sec)
[START] Episode 28


Episodes:  12%|█▏        | 29/240 [05:41<56:50, 16.16s/it]  

[START] Episode 4
Sim Complete
[DONE]  Episode 4 (Elapsed: 113.98 sec)
[START] Episode 15
Sim Complete
[DONE]  Episode 15 (Elapsed: 5.08 sec)
[START] Episode 24
Sim Complete
[DONE]  Episode 24 (Elapsed: 117.14 sec)
[START] Episode 35


...

Sim Complete
[DONE]  Episode 199 (Elapsed: 120.81 sec)
[START] Episode 212
Sim Complete
[DONE]  Episode 212 (Elapsed: 118.67 sec)
[START] Episode 225
Sim Complete
[DONE]  Episode 225 (Elapsed: 117.86 sec)
[START] Episode 238


Episodes: 100%|█████████▉| 239/240 [36:53<00:10, 10.04s/it]

Sim Complete
[DONE]  Episode 201 (Elapsed: 118.11 sec)
[START] Episode 213
Sim Complete
[DONE]  Episode 213 (Elapsed: 114.59 sec)
[START] Episode 226
Sim Complete
[DONE]  Episode 226 (Elapsed: 114.49 sec)
[START] Episode 239


Episodes: 100%|██████████| 240/240 [36:54<00:00,  9.23s/it]


Sim Complete
[DONE]  Episode 188 (Elapsed: 108.45 sec)
[START] Episode 202
Sim Complete
[DONE]  Episode 202 (Elapsed: 110.98 sec)
[START] Episode 214
Sim Complete
[DONE]  Episode 214 (Elapsed: 117.19 sec)
[START] Episode 227
Sim Complete
[DONE]  Episode 227 (Elapsed: 105.09 sec)
Sim Complete
[DONE]  Episode 198 (Elapsed: 119.38 sec)
[START] Episode 211
Sim Complete
[DONE]  Episode 211 (Elapsed: 123.33 sec)
[START] Episode 224
Sim Complete
[DONE]  Episode 224 (Elapsed: 54.63 sec)
[START] Episode 228
Sim Complete
[DONE]  Episode 228 (Elapsed: 114.48 sec)
Sim Complete
[DONE]  Episode 203 (Elapsed: 111.83 sec)
[START] Episode 215
Sim Complete
[DONE]  Episode 215 (Elapsed: 116.54 sec)
[START] Episode 229
Sim Complete
[DONE]  Episode 229 (Elapsed: 109.92 sec)
Sim Complete
[DONE]  Episode 231 (Elapsed: 109.42 sec)
Sim Complete
[DONE]  Episode 204 (Elapsed: 111.74 sec)
[START] Episode 216
Sim Complete
[DONE]  Episode 216 (Elapsed: 120.31 sec)
[START] Episode 230
Sim Complete
[DONE]  Episode 23

In [22]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(mrdp_episode_logs)]
mrdp_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/mrdp_{env_type}_{episode_num}_logs.parquet"
mrdp_episode_logs.write_parquet(file_name)

In [23]:
from decision import Agent, run_episodes_parallel

oracle_agent_kwargs = {
    'algorithm': 'oracle',
    # Optional: override horizon and action resolution for the oracle
    'soc_resolution': 20,
    'action_resolution': 41,  # number of discrete actions between -1 and 1
    'horizon': 48,  # plan 24 steps ahead (default: same as SDP horizon)

}

# Run all episodes in parallel using the oracle agent
oracle_episode_logs = run_episodes_parallel(
    Agent,
    selected_list[46:98],  # your list of environments
    agent_kwargs=oracle_agent_kwargs,
    max_workers=14,
    use_notebook_tqdm=False
)

[INFO] Starting 52 episodes with max_workers=14


Episodes: 100%|██████████| 52/52 [9:43:39<00:00, 673.45s/it]    


[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 9600.20 sec)
[START] Episode 26
Sim Complete
[DONE]  Episode 26 (Elapsed: 9499.56 sec)
[START] Episode 40
Sim Complete
[DONE]  Episode 40 (Elapsed: 8859.58 sec)
[START] Episode 2
Sim Complete
[DONE]  Episode 2 (Elapsed: 8823.94 sec)
[START] Episode 19
Sim Complete
[DONE]  Episode 19 (Elapsed: 9585.15 sec)
[START] Episode 36
Sim Complete
[DONE]  Episode 36 (Elapsed: 9555.40 sec)
[START] Episode 0
Sim Complete
[DONE]  Episode 0 (Elapsed: 9516.91 sec)
[START] Episode 24
Sim Complete
[DONE]  Episode 24 (Elapsed: 10132.37 sec)
[START] Episode 41
Sim Complete
[DONE]  Episode 41 (Elapsed: 8628.82 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 9152.14 sec)
[START] Episode 21
Sim Complete
[DONE]  Episode 21 (Elapsed: 9873.56 sec)
[START] Episode 38
Sim Complete
[DONE]  Episode 38 (Elapsed: 9527.09 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 8119.13 sec)
[START] Episode 15
Sim Complete
[DONE]  Epi

In [24]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(oracle_episode_logs)]
oracle_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/oracle_{env_type}_{episode_num}_46-98_logs.parquet"
oracle_episode_logs.write_parquet(file_name)

In [ ]:
from decision import Agent, run_sb3_model_on_vec_env
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from helper import flatten_episode_data

# (only needed if you ever switch to SubprocVecEnv on Linux/notebooks)
multiprocessing.set_start_method("forkserver", force=True)

# Utility to yield batches from a list
def batchify(lst, batch_size):
    """Yield successive batches from lst of size batch_size."""
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

from stable_baselines3.common.vec_env import SubprocVecEnv

batch_size = 64  # Set your desired batch size

In [ ]:
sac_model = SAC.load("../models/sac_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(sac_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

sac_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/sac_{env_type}_{episode_num}_logs.parquet"
sac_logs.write_parquet(file_name)

In [ ]:
PPO_model = PPO.load("../models/ppo_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(PPO_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

ppo_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ppo_{env_type}_{episode_num}_logs.parquet"
ppo_logs.write_parquet(file_name)

In [ ]:
a2c_model = A2C.load("../models/a2c_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(a2c_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

a2c_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/a2c_{env_type}_{episode_num}_logs.parquet"
a2c_logs.write_parquet(file_name)

In [ ]:
ddpg_model = DDPG.load("../models/ddpg_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(ddpg_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
ddpg_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ddpg_{env_type}_{episode_num}_logs.parquet"
ddpg_logs.write_parquet(file_name)

In [ ]:
td3_model = TD3.load("../models/td3_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(td3_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
td3_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/td3_{env_type}_{episode_num}_logs.parquet"
td3_logs.write_parquet(file_name)

In [ ]:
# to get the best RTG value, analysis of the distribution of total episode rewards in the dataset is needed


In [ ]:
from decision import Agent, run_episodes_parallel, run_single
from decision_transformer import DecisionTransformer
import json

import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

with open('../models/decision_transformer_model_kwargs.json', 'r') as f:
    model_kwargs = json.load(f)

model = DecisionTransformer(**model_kwargs)

model.load_state_dict(torch.load('../models/dt_model.pt', map_location=device))
model.return_scale = 1.0  # or whatever was used during training
model.eval()
rtg = 4508964.69


dt_agent_kwargs = {
    'algorithm': 'dt',
    'model': model.to(device),
    'rtg_value': rtg
}

# check for nan in model parameters
import math
bad_params = [name for name, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
print("bad params:", bad_params)

In [ ]:
episode_log = run_episodes_parallel(Agent, test_envs, agent_kwargs=dt_agent_kwargs, max_workers=2, use_notebook_tqdm=False)

dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_log)]
dt_logs = pl.concat(dfs_with_id)
file_name = f"../data/dt_rtg{int(rtg)}_{env_type}_{episode_num}_logs.parquet"
dt_logs.write_parquet(file_name)